# Data Simulation
This requires local copy of [SKA SDP E2E Batch continuum](https://gitlab.com/ska-telescope/sdp/science-pipeline-workflows/ska-sdp-e2e-batch-continuum-imaging) pipeline.
For data simulation e2e integration test [data setup](https://gitlab.com/ska-telescope/sdp/science-pipeline-workflows/ska-sdp-e2e-batch-continuum-imaging/-/blob/main/tests/e2e/resources/_data_sim.py?ref_type=heads#L124) was used with updated cable length delays.
```
Cable Length Delays: [-0.53192384, -0.44124044,  0.70160591, -0.17806597, -0.57407157,
 -0.35447444, -0.60360009, -0.42261453,  0.39747587,  0.23020702,
 -0.24653245, -0.28434663, -0.21342193,  0.52895497,  0.73860902,
  0.067399  , -0.14978879, -0.16425175]
```
Simulation configuration apart from the cable delays were the same as [tests/e2e/resources/_data_sim.py](https://gitlab.com/ska-telescope/sdp/science-pipeline-workflows/ska-sdp-e2e-batch-continuum-imaging/-/blob/main/tests/e2e/resources/_data_sim.py)

Update the cable lenght errors as per requirement in [ska-sdp-e2e-batch-continuum-imaging/-/blob/main/tests/e2e/resources/SKA-Low_AA2_18S_rigid-rotation_model.tm/cable_length_error.txt](https://gitlab.com/ska-telescope/sdp/science-pipeline-workflows/ska-sdp-e2e-batch-continuum-imaging/-/blob/main/tests/e2e/resources/SKA-Low_AA2_18S_rigid-rotation_model.tm/cable_length_error.txt?ref_type=heads)

For running the test the following commands were used:
```bash
$> docker run --rm -it -v "/path/to/ska-sdp-e2e-batch-continuum-imaging":/e2e/ -v /path/to/output/:/sim_output/ registry.gitlab.com/ska-telescope/sdp/ska-sdp-spack/ska-sdp-spack-ubuntu:2026.05.1-dev.cd1851d0f  /bin/bash

root@9bafa5081f51:~$ PYTHONPATH=/e2e/tests/ python
```

```python
>> from e2e.resources._data_sim import generate_calibrator_data
>> generate_calibrator_data("/sim_output", "FIELD_ID", "INTENT")
```

The simulated data would be present in `/path/to/output/test_visibility.ms`

# Setup

In [ ]:
from ska_sdp_instrumental_calibration.scheduler import UpstreamOutput
from ska_sdp_instrumental_calibration.stages import (
    load_data_stage, predict_vis_stage
)

from ska_sdp_instrumental_calibration.numpy_processors.solvers import Solver
from ska_sdp_instrumental_calibration.plot import PlotGaintableFrequency
from ska_sdp_instrumental_calibration.xarray_processors._utils import parse_antenna
from ska_sdp_instrumental_calibration.xarray_processors.solver import run_solver
from ska_sdp_instrumental_calibration.stages._utils import get_plots_path
from ska_sdp_instrumental_calibration.xarray_processors.delay import apply_delay, calculate_delay
from ska_sdp_instrumental_calibration.plot import plot_station_delays, PlotGaintableFrequency
from ska_sdp_instrumental_calibration.xarray_processors import with_chunks

import numpy as np
import xarray as xr
import dask
import matplotlib.pyplot as plt

In [ ]:
import pandas as pd

def unstack_jones_coordinate(ref_gaintable, gaintable):
    stack_gains = gaintable.gain.data
    new_gain_data = ref_gaintable.gain.data.copy()
    new_gain_data[..., 0, 0] = stack_gains[..., 0]
    new_gain_data[..., 1, 1] = stack_gains[..., 1]

    new_gain = ref_gaintable.gain.copy()
    new_gain.data = new_gain_data

    return ref_gaintable.assign(
        {
            "gain": new_gain,
        }
    ).chunk(ref_gaintable.chunks)
    
    
def stack_jones_coordinate(gaintable):
    gaintable = gaintable.stack(Jones_Solutions=("receptor1", "receptor2"))

    polstrs = [
        f"J_{p1}{p2}".upper()
        for p1, p2 in gaintable["Jones_Solutions"].data
    ]
    gaintable = gaintable.drop_vars(['Jones_Solutions', 'receptor1', 'receptor2']).assign_coords({"Jones_Solutions": polstrs})

    return gaintable

# Delay Calibration

Pipeline Structure:
1. Load Data
2. Delay Calibration
3. ...


Gain substitution with [solve_antenna_gains_itsubs_scalar](https://gitlab.com/ska-telescope/sdp/ska-sdp-func-python/-/blob/main/src/ska_sdp_func_python/calibration/solvers.py?ref_type=heads#L408) was used to solve for bandpass. The delays were extracted from the solved gaintables. They were ploted and compared against the cable length delays.

## Approach
`solve_antenna_gains_itsubs_scalar` works on single polarization, hence the run_solver was run as a map-reduce on `["XX", "YY"]`.

In [ ]:
def delay_calibration(
    upstream_output,
    output_path,
    refant = 0,
    niter = 200,
    tol  = 1e-06,
):
    vis = upstream_output.vis
    modelvis = upstream_output.modelvis
    initialtable = upstream_output.gaintable
    prefix = upstream_output.ms_prefix

    refant = parse_antenna(refant, initialtable.configuration.names)
    solver = Solver.get_solver(refant=refant, niter=niter, tol=tol) 

    # `solve_antenna_gains_itsubs_scalar` works on single polarization, 
    # Map-reduce run_solver on `["XX", "YY"]`
    
    gaintables = []
    pols = ["XX", "YY"]

    for pol in pols:
        scalar_vis = vis.sel(polarisation=[pol])
        scalar_model_vis = modelvis.sel(polarisation=[pol])
        scalar_table = initialtable.sel(receptor1=[pol[0]], receptor2=[pol[0]])
        scalar_gaintable = run_solver(  # update
            vis=scalar_vis,
            modelvis=scalar_model_vis,
            gaintable=scalar_table,
            solver=solver,
        )
    
        gaintables.append(scalar_gaintable.compute())

    gaintable = unstack_jones_coordinate(
        initialtable, 
        xr.merge([stack_jones_coordinate(table) for table in gaintables])
    )
    
    delaytable = calculate_delay(gaintable, oversample=1)
    corrected_gaintable = apply_delay(gaintable, delaytable)
    
    prefix = upstream_output.ms_prefix
    path_prefix = get_plots_path(
            output_path, f"{prefix}/delay"
        )

    upstream_output.add_compute_tasks(
        plot_station_delays(
            delaytable,
            path_prefix,
        )
    )

    freq_plotter = PlotGaintableFrequency(
        path_prefix=path_prefix,
        refant=refant,
    )

    upstream_output.add_compute_tasks(
        *freq_plotter.plot(
                gaintable,
                figure_title="Delay",
                fixed_axis=False,
        )
    )

    freq_plotter = PlotGaintableFrequency(
        path_prefix=f"{path_prefix}_corrected",
        refant=refant,
    )
    
    upstream_output.add_compute_tasks(
        *freq_plotter.plot(
                corrected_gaintable,
                figure_title="Delay Corrected",
                fixed_axis=False,
        )
    )
    
    upstream_output["gaintable"] = gaintable
    upstream_output["refant"] = refant
    upstream_output["delaytable"] = delaytable

    return upstream_output

# Run Pipeline

In [ ]:
input_ms = ["/data/SKA/inst_data/test_delay_visbility_240.ms"]
lsm_csv = "/data/SKA/inst_data/local_sky_model.csv"
output_dir = "./delay_cal_spike_output"

upstream_output = UpstreamOutput()

In [ ]:
upstream_output = load_data_stage(
    upstream_output,
    output_dir,
    input_ms=input_ms,
    cache_directory="./cache",
)

upstream_output = predict_vis_stage(
    upstream_output[0], output_dir, input_ms=input_ms, lsm_csv_path=lsm_csv
)

upstream_output = delay_calibration(
    upstream_output,
    output_dir
)

gaintable = upstream_output.gaintable.compute()

In [ ]:
dask.compute(*upstream_output.compute_tasks)

# Output Plots

In [ ]:
from IPython.display import Markdown, display
ms_name = upstream_output.ms_prefix
for plot in ["delay_station_delay", "delay-gain-phase-freq", "delay_corrected-gain-phase-freq"]:
    display(Markdown(f"![Delay Corrected Phase Plot](./{output_dir}/plots/{ms_name}/{plot}.png)"))

# Validation
After the pipeline has finished, we need to compare the delays from the pipeline with the cable lenght delays used in the data simulation in meters. We need to convert the delays from meters to seconds using the following formula:

$$
\mathrm{delay}_{\mathrm{ns}} = \frac{\mathrm{length}_{\mathrm{m}}}{c}
$$

where $c = 299792458 \text{ m/s}.$

**Verification**
1. The delay plots should match.
2. The phase in the gaintable plots, after applying delay effect, should be zero.

In [ ]:
simulation_station_delays = np.array([-0.53192384, -0.44124044,  0.70160591, -0.17806597, -0.57407157,
       -0.35447444, -0.60360009, -0.42261453,  0.39747587,  0.23020702,
       -0.24653245, -0.28434663, -0.21342193,  0.52895497,  0.73860902,
        0.067399  , -0.14978879, -0.16425175]) / 299782458

simulation_ref_delays = (simulation_station_delays - simulation_station_delays[upstream_output.refant])

delaytable = upstream_output["delaytable"]
station_name = delaytable.configuration.names.data
calibration_delay = delaytable.delay.data 
cal_delay_data = calibration_delay[..., 0].reshape(len(station_name))

np.testing.assert_almost_equal(cal_delay_data, simulation_ref_delays, decimal=9)